# Spatio-Temporal Graph Neural Network (ST-GNN) Demo (Core)

This notebook defines and trains a simple ST-GNN (GNN + LSTM) on dummy data.
Sanity checks have been moved to a standalone script: `pytorch_geometric_sanity_check.py`.


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import numpy as np


## Define the Spatio-Temporal GNN
This model uses a GATConv stack for spatial processing, followed by an LSTM for temporal dynamics, and a linear layer for regression.

In [ ]:
class STGNN(nn.Module):
    def __init__(self, in_features, hidden_features, out_features, heads=4):
        super(STGNN, self).__init__()
        self.gnn1 = GATConv(in_features, hidden_features, heads=heads, concat=True)
        self.gnn2 = GATConv(hidden_features*heads, hidden_features)
        self.lstm = nn.LSTM(hidden_features, hidden_features, batch_first=True)
        self.fc = nn.Linear(hidden_features, out_features)

    def forward(self, x_seq, edge_index):
        batch_size, seq_len, n_nodes, in_features = x_seq.shape
        embeddings = []
        for t in range(seq_len):
            x_t = x_seq[:, t, :, :]
            x_t = x_t.reshape(-1, in_features)
            h = self.gnn1(x_t, edge_index).relu()
            h = self.gnn2(h, edge_index).relu()
            h = h.reshape(batch_size, n_nodes, -1)
            embeddings.append(h)
        embeddings = torch.stack(embeddings, dim=1)
        embeddings = embeddings.permute(0,2,1,3).reshape(batch_size*n_nodes, seq_len, -1)
        lstm_out, _ = self.lstm(embeddings)
        final_state = lstm_out[:, -1, :]
        out = self.fc(final_state)
        out = out.reshape(batch_size, n_nodes, -1)
        return out

## Dummy Data & Training Loop
We create fake time-series features and train the model for a few epochs just to demonstrate the workflow.

In [ ]:
n_nodes = 5
in_features = 6
hidden_features = 16
out_features = 1

# Fake input: batch=2, seq_len=10
x_seq = torch.randn(2, 10, n_nodes, in_features)

# Simple chain graph connectivity (undirected)
edge_index = torch.tensor([[0,1,2,3],[1,2,3,4]], dtype=torch.long)
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

model = STGNN(in_features, hidden_features, out_features)
target = torch.randn(2, n_nodes, out_features)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    optimizer.zero_grad()
    pred = model(x_seq, edge_index)
    loss = criterion(pred, target)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")